In [ ]:
import torch
import torch.nn.functional as F
import os
from vllm import LLM, SamplingParams
os.environ.setdefault("VLLM_USE_V1", "1")
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

llm = LLM(model="meta-llama/Meta-Llama-3.1-8B-Instruct", enforce_eager=True, tensor_parallel_size=1)
engine = llm.llm_engine


In [ ]:
# Robust KV cache extraction that works with vLLM v1 API
req_id = "kv_analysis5"
prompt = "Explain how transformer attention mechanisms work with key-value caches."

# Add request and process it
engine.add_request(req_id, prompt, SamplingParams(max_tokens=25))
print("Processing request...")

# Process the request completely
outputs = []
for i in range(10):
    result = engine.step()
    if result:
        outputs.extend(result)
        print(f"Step {i+1}: Generated {len(result)} outputs")
        # Check if our request is complete
        for output in result:
            if output.request_id == req_id and output.finished:
                print(f"Request {req_id} completed!")
                break
    else:
        print(f"Step {i+1}: No outputs")
        break

print("\nRequest processing completed.")

# Get the engine core and scheduler
core_client = engine.engine_core
if hasattr(core_client, "engine_core"):
    engine_core = core_client.engine_core
else:
    raise RuntimeError("This notebook expects an in-process EngineCore.")

scheduler = engine_core.scheduler
print(f"\nScheduler state:")
print(f"  Running: {len(scheduler.running)}")
print(f"  Waiting: {len(scheduler.waiting)}")


# Access the KV cache directly from the model and attention layers
ctx = engine.vllm_config.compilation_config.static_forward_context
print(f"\nAvailable attention layers:")
attn_layers = {}
for key in ctx.keys():
    if "self_attn.attn" in key:
        layer_num = int(key.split('.')[2])  # Extract layer number
        attn_layers[layer_num] = ctx[key]
        print(f"  Layer {layer_num}: {key}")

layer_a, layer_b = 1, 24

layer_early = attn_layers[layer_a]
layer_late = attn_layers[layer_b]

print(f"\nAnalyzing layers {layer_a+1} and {layer_b+1}:")
print(f"  Early layer: {layer_early}")
print(f"  Late layer: {layer_late}")

# Inspect KV cache structure
print(f"\nKV Cache structure:")
if hasattr(layer_early, 'kv_cache'):
    kv_cache = layer_early.kv_cache
    print(f"  KV cache type: {type(kv_cache)}")
    print(f"  KV cache length: {len(kv_cache)}")
    
    if len(kv_cache) > 0:
        ve_0 = kv_cache[0]  # Virtual engine 0
        print(f"  Virtual engine 0 type: {type(ve_0)}")
        print(f"  Virtual engine 0 length: {len(ve_0)}")
        
        if len(ve_0) >= 2:
            key_cache = ve_0[0]
            value_cache = ve_0[1]
            print(f"  Key cache shape: {key_cache.shape}")
            print(f"  Value cache shape: {value_cache.shape}")
            print(f"  Key cache dtype: {key_cache.dtype}")
            print(f"  Value cache dtype: {value_cache.dtype}")
else:
    print("  No kv_cache attribute found")

def extract_kv_from_blocks(attn_layer, max_tokens=50, ve=0):
    """
    Extract KV cache data from attention layer blocks
    
    Args:
        attn_layer: The attention layer
        max_tokens: Maximum number of tokens to extract
        ve: Virtual engine index
    
    Returns:
        tuple: (keys, values) tensors of shape [seq_len, hidden_dim]
    """
    if not hasattr(attn_layer, 'kv_cache'):
        raise RuntimeError("No kv_cache found in attention layer")
    
    kv_cache = attn_layer.kv_cache[ve]
    key_cache = kv_cache[0]  # [num_blocks, block_size, num_kv_heads, head_dim]
    value_cache = kv_cache[1]  # [num_blocks, block_size, num_kv_heads, head_dim]
    
    print(f"    Cache shapes - Key: {key_cache.shape}, Value: {value_cache.shape}")
    
    # Find non-empty blocks (blocks with actual data)
    # We'll look for blocks that have non-zero data
    num_blocks, block_size, num_heads, head_dim = key_cache.shape

    print(key_cache.reshape(-1))
    
    # Extract data from the first few blocks (assuming sequential filling)
    max_blocks = min(num_blocks, (max_tokens + block_size - 1) // block_size)
    
    # Concatenate blocks
    keys_blocks = key_cache[:max_blocks]  # [max_blocks, block_size, num_heads, head_dim]
    values_blocks = value_cache[:max_blocks]  # [max_blocks, block_size, num_heads, head_dim]
    
    # Reshape to sequence format
    keys_seq = keys_blocks.reshape(-1, num_heads, head_dim)[:max_tokens]  # [seq_len, num_heads, head_dim]
    values_seq = values_blocks.reshape(-1, num_heads, head_dim)[:max_tokens]  # [seq_len, num_heads, head_dim]
    
    # Flatten to [seq_len, hidden_dim]
    keys_flat = keys_seq.reshape(keys_seq.shape[0], -1).float().cpu()
    values_flat = values_seq.reshape(values_seq.shape[0], -1).float().cpu()
    
    print(f"    Extracted shapes - Keys: {keys_flat.shape}, Values: {values_flat.shape}")
    
    return keys_flat, values_flat

# Extract KV caches from both layers
print(f"\n=== Extracting KV caches ===")
print(f"Early layer (Layer {layer_a+1}):")
keys_early, values_early = extract_kv_from_blocks(layer_early)

print(f"\nLate layer (Layer {layer_b+1}):")
keys_late, values_late = extract_kv_from_blocks(layer_late)

# Ensure same sequence length for comparison
min_len = min(keys_early.shape[0], keys_late.shape[0])
keys_early = keys_early[:min_len]
keys_late = keys_late[:min_len]
values_early = values_early[:min_len]
values_late = values_late[:min_len]

print(f"\n=== Computing similarities for {min_len} tokens ===")

# Compute token-wise similarities
key_similarities = F.cosine_similarity(keys_early, keys_late, dim=-1)
value_similarities = F.cosine_similarity(values_early, values_late, dim=-1)

print(f"Key similarities shape: {key_similarities.shape}")
print(f"Value similarities shape: {value_similarities.shape}")

# Statistics
print(f"\nKey Cache Similarities (Layer {layer_a+1} vs Layer {layer_b+1}):")
print(f"  Mean: {key_similarities.mean().item():.4f}")
print(f"  Std:  {key_similarities.std().item():.4f}")
print(f"  Min:  {key_similarities.min().item():.4f}")
print(f"  Max:  {key_similarities.max().item():.4f}")

print(f"\nValue Cache Similarities (Layer {layer_a+1} vs Layer {layer_b+1}):")
print(f"  Mean: {value_similarities.mean().item():.4f}")
print(f"  Std:  {value_similarities.std().item():.4f}")
print(f"  Min:  {value_similarities.min().item():.4f}")
print(f"  Max:  {value_similarities.max().item():.4f}")
